# Databricks Asset Bundles (DABs) & GitHub Actions CI/CD Guide

A complete end-to-end reference and walkthrough for deploying the **Sales Medallion Architecture (`sales_dabs`)** across `dev`, `uat`, and `prod` environments using **OAuth U2M**, **OAuth M2M / OIDC**, and **GitHub Actions**.

---

## Part 1: Local Development Setup (OAuth U2M)

Used for human-driven local development and testing in the `dev` target.

### 1. Create Profile & Authenticate via U2M
Log in interactively via your browser and bind the session to a named CLI profile (`Divi`):

```powershell
# Authenticate via browser and create profile 'Divi'
databricks auth login --host https://dbc-14af95b7-ea77.cloud.databricks.com --profile Divi

# Verify active user identity
databricks current-user me --profile Divi
```

### 2. Validate & Deploy Bundle in `dev`
Run local validation and deploy the isolated development resources (`[dev] sales_dabs_job`):

```powershell
# 1. Validate YAML syntax, variables, and notebook paths
databricks bundle validate -t dev --profile Divi

# 2. Deploy bundle files and job to dev workspace
databricks bundle deploy -t dev --profile Divi

# 3. Execute the 4-step Medallion job (00_setup -> 01_bronze_ingest -> 02_silver_clean -> 03_gold_kpis)
databricks bundle run sales_dabs_job -t dev --profile Divi
```

---

## Part 2: Production Identity & Governance (Service Principal + M2M)

Production jobs must run under a non-human **Service Principal** rather than an individual developer's account.

### 1. Create the Service Principal (`sales_prod_sp`)
1. Go to **Databricks Settings (top right) → Identity and access → Service principals → Add service principal**.
2. Name it `sales_prod_sp` and click **Add**.
3. Copy its **Application ID** (UUID) and generate an **OAuth Secret** under its **Secrets** tab.

### 2. Configure `run_as` in `databricks.yml`
Set `run_as` under `uat` and `prod` targets using `sales_prod_sp`'s Application ID:

```yaml
variables:
  catalog:
    default: dev
  sp_application_id:
    default: "<SALES_PROD_SP_APPLICATION_ID>"

targets:
  dev:
    mode: development
    default: true
    workspace:
      host: https://dbc-14af95b7-ea77.cloud.databricks.com
      profile: Divi
    variables:
      catalog: dev

  uat:
    mode: production
    workspace:
      host: https://dbc-14af95b7-ea77.cloud.databricks.com
      root_path: /Workspace/Users/divyankapal22@gmail.com/.bundle/${bundle.name}/${bundle.target}
    run_as:
      service_principal_name: ${var.sp_application_id}
    variables:
      catalog: uat

  prod:
    mode: production
    workspace:
      host: https://dbc-14af95b7-ea77.cloud.databricks.com
      root_path: /Workspace/Users/divyankapal22@gmail.com/.bundle/${bundle.name}/${bundle.target}
    run_as:
      service_principal_name: ${var.sp_application_id}
    variables:
      catalog: prod
```

### 3. Grant Required Permissions
Complete these three permission grants before running CI/CD:

1. **Unity Catalog (`uat` and `prod` Catalogs):**
   * Grant `sales_prod_sp` **`ALL PRIVILEGES`** on both `uat` and `prod` catalogs so it can create schemas (`bronze`, `silver`, `gold`), volumes, and tables.
2. **Service Principal User Access:**
   * Go to **Settings → Identity and access → Service principals → `sales_prod_sp` → Permissions**.
   * Grant your user (`divyankapal22@gmail.com`) **Service Principal: Manager / User** permissions so you are authorized to deploy jobs configured with `run_as: sales_prod_sp`.
3. **Workspace Bundle Folder Permissions:**
   * **Option A (User Folder):** Navigate to `/Workspace/Users/divyankapal22@gmail.com/.bundle` and `.bundle/sales_dabs` → click **Share** → grant `sales_prod_sp` **`Can Manage`** so it can create and update target directories (`uat`, `prod`).
   * **Option B (Shared Folder — Safe Alternative):** Use `/Workspace/Shared/.bundle/${bundle.name}/${bundle.target}` as `root_path` if you want to avoid granting permissions inside personal user folders.

---

## Part 3: GitHub Actions CI/CD Setup

### 1. Verify Git Repository Root & Create `.github/workflows/`
GitHub Actions **only** detects workflow files placed inside `.github/workflows/` at the **top-level root** of your Git repository (never inside a subfolder):

```powershell
# Confirm top-level Git root
git rev-parse --show-toplevel

# Create workflows directory at repo root
New-Item -ItemType Directory -Force -Path ".github\workflows"
```

### 2. Configure Secrets in GitHub Repository
Open your GitHub repository → **Settings → Secrets and variables → Actions → Secrets** and add:

| Secret Name | Value |
| :--- | :--- |
| `DATABRICKS_HOST` | `https://dbc-14af95b7-ea77.cloud.databricks.com` |
| `DATABRICKS_CLIENT_ID` | `<sales_prod_sp Application ID>` |
| `DATABRICKS_CLIENT_SECRET` | `<sales_prod_sp OAuth Secret>` |

### 3. Create Workflow 1: PR Validation (`.github/workflows/validate_sales_dab.yml`)
* **Trigger:** Opening or updating a Pull Request targeting `main`.
* **Action:** Authenticates via `oauth-m2m` and runs `databricks bundle validate` against `uat` and `prod` **without deploying** any resources.

```yaml
name: 1. Validate Sales DAB (M2M)

on:
  pull_request:
    branches:
      - main

jobs:
  validate:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout Repository
        uses: actions/checkout@v4

      - name: Install Databricks CLI
        uses: databricks/setup-cli@main

      - name: Validate UAT Bundle (M2M)
        env:
          DATABRICKS_AUTH_TYPE: oauth-m2m
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_CLIENT_ID: ${{ secrets.DATABRICKS_CLIENT_ID }}
          DATABRICKS_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}
        run: databricks bundle validate -t uat

      - name: Validate Prod Bundle (M2M)
        env:
          DATABRICKS_AUTH_TYPE: oauth-m2m
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_CLIENT_ID: ${{ secrets.DATABRICKS_CLIENT_ID }}
          DATABRICKS_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}
        run: databricks bundle validate -t prod
```

### 4. Create Workflow 2: Production Deployment (`.github/workflows/deploy_sales_dab.yml`)
* **Trigger:** Merging a Pull Request into `main`.
* **Action:** Validates, deploys `[prod] sales_dabs_job`, and executes the end-to-end ETL pipeline in production under `sales_prod_sp`. *(Note: Can also be upgraded to passwordless `github-oidc` with `id-token: write` once a GitHub Federation Policy is attached to `sales_prod_sp`.)*

```yaml
name: 2. Deploy Sales DAB to Prod (CI/CD)

on:
  push:
    branches:
      - main

jobs:
  deploy-prod:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout Repository
        uses: actions/checkout@v4

      - name: Install Databricks CLI
        uses: databricks/setup-cli@main

      - name: Validate Production Bundle
        env:
          DATABRICKS_AUTH_TYPE: oauth-m2m
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_CLIENT_ID: ${{ secrets.DATABRICKS_CLIENT_ID }}
          DATABRICKS_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}
        run: databricks bundle validate -t prod

      - name: Deploy Bundle to Production
        env:
          DATABRICKS_AUTH_TYPE: oauth-m2m
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_CLIENT_ID: ${{ secrets.DATABRICKS_CLIENT_ID }}
          DATABRICKS_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}
        run: databricks bundle deploy -t prod

      - name: Run Production Job (Proof of Execution)
        env:
          DATABRICKS_AUTH_TYPE: oauth-m2m
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_CLIENT_ID: ${{ secrets.DATABRICKS_CLIENT_ID }}
          DATABRICKS_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}
        run: databricks bundle run sales_dabs_job -t prod
```

---

## Part 4: Execution Flow

1. **Push Changes to Feature Branch:**
   ```powershell
   git checkout -b feature/sales-medallion-dabs
   git add -A
   git commit -m "Add Sales Medallion DABs and CI/CD workflows"
   git push -u origin feature/sales-medallion-dabs
   ```
2. **Create Pull Request (`feature/sales-medallion-dabs` → `main`):**
   * Open GitHub → **Compare & pull request**.
   * Navigate to the **Actions** tab and show **`1. Validate Sales DAB (M2M)`** passing—confirming `uat` and `prod` configurations are valid without touching production jobs.
3. **Merge Pull Request to `main`:**
   * Click **Merge pull request**.
   * Watch **`2. Deploy Sales DAB to Prod (CI/CD)`** trigger automatically in GitHub Actions.
4. **Verify Live Production Execution in Databricks:**
   * Open **Workflows → `[prod] sales_dabs_job`** → verify **Run as: `sales_prod_sp`** and **Parameter: `catalog = prod`**.
   * Open **Catalog Explorer → `prod`** → show populated tables in `prod.bronze` (`sales_raw`), `prod.silver` (`sales_cleaned`), and `prod.gold` (`kpi_regional_sales`, `kpi_monthly_revenue`, `kpi_top_products`, `kpi_customer_ltv`).

---

## Part 5: Emergency Production Rollback Plan

If a production deployment introduces an issue, revert cleanly using Git version control:

```powershell
# 1. Identify the last known working commit hash
git log --oneline

# 2. Check out the stable commit
git checkout <previous-working-commit-hash>

# 3. Validate and redeploy the previous working state to prod
databricks bundle validate -t prod
databricks bundle deploy -t prod

# 4. Return to main branch once production is verified
git checkout main
```

## Bonus: Passwordless Production CI/CD with GitHub OIDC

> **Edition Note:** OIDC Federation Policies require **Databricks Enterprise / Account Console access** (not available in Databricks Free Edition, where `oauth-m2m` with GitHub Secrets is used instead).

### How OIDC Differs from M2M
* **Zero Secrets in GitHub:** Eliminates `DATABRICKS_CLIENT_SECRET` completely—no secret rotation or expiration risk.
* **Short-Lived Tokens:** GitHub mints a single-use signed JWT per workflow run (`id-token: write`), which Databricks verifies against a Service Principal Federation Policy to issue a temporary access token.

### Setup Steps (Brief)
1. **Create Federation Policy (Databricks Account Console):**
   * Go to `accounts.cloud.databricks.com` → **User management → Service principals → `sales_prod_sp` → Credentials & secrets → Federation policies → Create policy**.
   * Set Provider to **GitHub Actions**, Org to `divyankapal22`, Repository to `sales_medallion`, and Branch to `main`.
2. **Add Plain Variables in GitHub (No Secrets):**
   * **Settings → Secrets and variables → Actions → Variables:** add `DATABRICKS_HOST` and `DATABRICKS_CLIENT_ID` (`sales_prod_sp` Application ID).
3. **Update Workflow (`deploy_sales_dab.yml`):**
   * Add `permissions: id-token: write` at the top level.
   * Set `DATABRICKS_AUTH_TYPE: github-oidc` and reference `${{ vars.DATABRICKS_HOST }}` and `${{ vars.DATABRICKS_CLIENT_ID }}`.